In [1]:
from pathlib import Path
import os
import zipfile
import shutil

import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
RAW_ZIP_PATH = Path("/content/trafficpulse_raw_datasets.zip")
EXTRACT_PATH = Path("/content")

if not RAW_ZIP_PATH.exists():
    uploaded_files = list(uploaded.keys())
    RAW_ZIP_PATH = Path("/content") / uploaded_files[0]

print("Using zip file:", RAW_ZIP_PATH)

with zipfile.ZipFile(RAW_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Unzipped successfully.")

Using zip file: /content/trafficpulse_raw_datasets.zip
Unzipped successfully.


In [3]:
for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".h5", ".csv", ".geojson", ".graphml")):
            print(Path(root) / file)

/content/final_merged_dataset.csv
/content/content/trafficpulse_dataset/weather/la_weather_metr_la_period.csv
/content/content/trafficpulse_dataset/dcrnn/metr-la.h5
/content/content/trafficpulse_dataset/dcrnn/pems-bay.h5
/content/content/trafficpulse_dataset/sensor_graph/graph_sensor_locations.csv
/content/content/trafficpulse_dataset/osm/la_road_nodes.geojson
/content/content/trafficpulse_dataset/osm/la_road_edges.geojson
/content/content/trafficpulse_dataset/osm/la_road_network.graphml
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

BASE_PATH = Path("/content/content/trafficpulse_dataset")

TRAFFIC_PATH = BASE_PATH / "dcrnn"
SENSOR_PATH = BASE_PATH / "sensor_graph"
WEATHER_PATH = BASE_PATH / "weather"
OSM_PATH = BASE_PATH / "osm"

PREPROCESSED_PATH = Path("/content/trafficpulse_preprocessed")
PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Traffic path:", TRAFFIC_PATH)
print("Sensor path:", SENSOR_PATH)
print("Weather path:", WEATHER_PATH)
print("OSM path:", OSM_PATH)
print("Output path:", PREPROCESSED_PATH)

Traffic path: /content/content/trafficpulse_dataset/dcrnn
Sensor path: /content/content/trafficpulse_dataset/sensor_graph
Weather path: /content/content/trafficpulse_dataset/weather
OSM path: /content/content/trafficpulse_dataset/osm
Output path: /content/trafficpulse_preprocessed


## Load Traffic dataset

In [5]:
traffic_path = TRAFFIC_PATH / "metr-la.h5"

traffic_wide = pd.read_hdf(traffic_path)

In [6]:
traffic_wide.sample(5)

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-19 19:45:00,66.111111,63.777778,65.444444,57.444444,58.666667,59.444444,61.555556,63.333333,65.333333,67.750,...,30.833333,69.444444,61.777778,61.666667,67.111111,65.111111,69.000,64.666667,64.444444,61.333333
2012-04-02 12:00:00,66.500000,63.375000,67.000000,52.375000,44.125000,58.875000,61.000000,0.000000,61.875000,66.375,...,44.375000,65.125000,63.500000,66.875000,66.125000,62.750000,66.250,61.000000,63.750000,63.875000
2012-04-01 00:30:00,67.000000,65.625000,69.000000,61.000000,65.250000,68.500000,65.375000,63.500000,65.875000,66.500,...,48.625000,69.000000,66.625000,67.500000,69.375000,62.250000,68.125,67.875000,69.375000,63.375000
2012-04-06 21:45:00,67.125000,64.625000,67.500000,55.125000,42.125000,52.000000,60.625000,61.625000,65.875000,66.375,...,47.875000,68.857143,66.125000,64.875000,67.250000,64.125000,69.125,64.375000,65.375000,61.142857
2012-05-22 02:05:00,61.750000,62.750000,68.000000,60.625000,57.125000,41.125000,59.750000,63.125000,55.750000,50.750,...,41.750000,70.000000,56.375000,57.250000,0.000000,0.000000,61.375,57.000000,65.875000,59.125000


In [7]:
traffic_wide.shape

(34272, 207)

In [8]:
(traffic_wide == 0)

,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-01 00:00:00,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-03-01 00:05:00,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-03-01 00:10:00,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-03-01 00:15:00,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
2012-03-01 00:20:00,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2012-06-27 23:35:00,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-06-27 23:40:00,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-06-27 23:45:00,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2012-06-27 23:50:00,False,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:
total_values = traffic_wide.shape[0] * traffic_wide.shape[1]

zero_count = (traffic_wide == 0).sum().sum()
zero_percent = zero_count / total_values * 100

all_zero_rows = (traffic_wide == 0).all(axis=1)

print("Total values:", total_values)
print("Zero values:", zero_count)
print("Zero percentage:", round(zero_percent, 4), "%")
print("Rows where all sensors are zero:", all_zero_rows.sum())

traffic_wide[all_zero_rows].head()

Total values: 7094304
Zero values: 575302
Zero percentage: 8.1094 %
Rows where all sensors are zero: 2148


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-03-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-03-03 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-03-05 13:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2012-03-05 13:35:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
all_zero_rows = (traffic_wide == 0).all(axis=1)

# Find consecutive all-zero blocks
zero_blocks = all_zero_rows.astype(int)

block_id = (zero_blocks != zero_blocks.shift()).cumsum()

block_lengths = zero_blocks.groupby(block_id).sum()

all_zero_block_lengths = block_lengths[block_lengths > 0]

print("Number of all-zero blocks:", len(all_zero_block_lengths))
print("Longest all-zero block length:", all_zero_block_lengths.max())
print("Longest all-zero block duration in minutes:", all_zero_block_lengths.max() * 5)

all_zero_block_lengths.describe()

Number of all-zero blocks: 97
Longest all-zero block length: 381
Longest all-zero block duration in minutes: 1905


,0
count,97.000000
mean,22.144330
std,52.457322
min,1.000000
25%,1.000000
50%,2.000000
75%,12.000000
max,381.000000


In [11]:
all_zero_rows = (traffic_wide == 0).all(axis=1)

print("Original traffic shape:", traffic_wide.shape)
print("All-zero timestamps:", all_zero_rows.sum())

traffic_wide_no_global_zero = traffic_wide.loc[~all_zero_rows].copy()

print("After dropping all-zero timestamps:", traffic_wide_no_global_zero.shape)

Original traffic shape: (34272, 207)
All-zero timestamps: 2148
After dropping all-zero timestamps: (32124, 207)


In [12]:
traffic_clean_wide = traffic_wide_no_global_zero.copy()

traffic_clean_wide[traffic_clean_wide <= 0] = np.nan

missing_count = traffic_clean_wide.isna().sum().sum()
missing_percent = missing_count / traffic_clean_wide.size * 100

print("Missing values after converting remaining zeros to NaN:", missing_count)
print("Missing percentage:", round(missing_percent, 4), "%")

Missing values after converting remaining zeros to NaN: 130666
Missing percentage: 1.965 %


In [13]:
traffic_clean_wide = traffic_clean_wide.interpolate(
    method="time",
    axis=0,
    limit=6,
    limit_direction="both"
)

traffic_clean_wide = traffic_clean_wide.ffill(limit=6).bfill(limit=6)

remaining_missing = traffic_clean_wide.isna().sum().sum()
remaining_missing_percent = remaining_missing / traffic_clean_wide.size * 100

print("Remaining missing values:", remaining_missing)
print("Remaining missing percentage:", round(remaining_missing_percent, 4), "%")

Remaining missing values: 83959
Remaining missing percentage: 1.2626 %


In [14]:
row_missing_ratio = traffic_clean_wide.isna().mean(axis=1)

print(row_missing_ratio.describe())

print("Rows with more than 20% missing sensors:", (row_missing_ratio > 0.20).sum())
print("Rows with more than 10% missing sensors:", (row_missing_ratio > 0.10).sum())
print("Rows with more than 5% missing sensors:", (row_missing_ratio > 0.05).sum())

count    32124.000000
mean         0.012626
std          0.015894
min          0.000000
25%          0.000000
50%          0.004831
75%          0.019324
max          0.101449
dtype: float64
Rows with more than 20% missing sensors: 0
Rows with more than 10% missing sensors: 4
Rows with more than 5% missing sensors: 1513


In [15]:
# Since no row has more than 20% missing sensors, this will keep all rows
traffic_clean_wide = traffic_clean_wide.loc[row_missing_ratio <= 0.20].copy()

print("Shape after row missing filter:", traffic_clean_wide.shape)
print("Remaining missing values:", traffic_clean_wide.isna().sum().sum())

Shape after row missing filter: (32124, 207)
Remaining missing values: 83959


In [16]:
traffic_long = traffic_clean_wide.reset_index()

traffic_long = traffic_long.rename(columns={
    traffic_long.columns[0]: "timestamp"
})

traffic_long = traffic_long.melt(
    id_vars="timestamp",
    var_name="sensor_id",
    value_name="speed"
)

traffic_long["timestamp"] = pd.to_datetime(traffic_long["timestamp"])
traffic_long["sensor_id"] = traffic_long["sensor_id"].astype(str)
traffic_long["speed"] = pd.to_numeric(traffic_long["speed"], errors="coerce")

print("Traffic long shape before dropping remaining NaN:", traffic_long.shape)
print("Missing speed before drop:", traffic_long["speed"].isna().sum())

traffic_long.head()

Traffic long shape before dropping remaining NaN: (6649668, 3)
Missing speed before drop: 83959


,timestamp,sensor_id,speed
0,2012-03-01 00:00:00,773869,64.375000
1,2012-03-01 00:05:00,773869,62.666667
2,2012-03-01 00:10:00,773869,64.000000
3,2012-03-01 00:25:00,773869,57.333333
4,2012-03-01 00:30:00,773869,66.500000


In [17]:
traffic_long = traffic_long.dropna(subset=["speed"]).reset_index(drop=True)

print("Traffic long shape after dropping remaining NaN:", traffic_long.shape)
print("Missing speed after drop:", traffic_long["speed"].isna().sum())

traffic_long.head()

Traffic long shape after dropping remaining NaN: (6565709, 3)
Missing speed after drop: 0


,timestamp,sensor_id,speed
0,2012-03-01 00:00:00,773869,64.375000
1,2012-03-01 00:05:00,773869,62.666667
2,2012-03-01 00:10:00,773869,64.000000
3,2012-03-01 00:25:00,773869,57.333333
4,2012-03-01 00:30:00,773869,66.500000


In [18]:
traffic_long.isnull().sum()

,0
timestamp,0
sensor_id,0
speed,0


In [19]:
traffic_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6565709 entries, 0 to 6565708
Data columns (total 3 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   sensor_id  object        
 2   speed      float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 150.3+ MB


## Loading sensor location dataset

In [20]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/content/trafficpulse_dataset")
SENSOR_PATH = BASE_PATH / "sensor_graph"

sensor_path = SENSOR_PATH / "graph_sensor_locations.csv"

print("Sensor file path:", sensor_path)
print("File exists:", sensor_path.exists())

Sensor file path: /content/content/trafficpulse_dataset/sensor_graph/graph_sensor_locations.csv
File exists: True


In [21]:
sensor_df = pd.read_csv(sensor_path)

print("Sensor location dataset loaded.")
print("Shape:", sensor_df.shape)
print("Columns:", sensor_df.columns.tolist())

sensor_df.head()

Sensor location dataset loaded.
Shape: (207, 4)
Columns: ['index', 'sensor_id', 'latitude', 'longitude']


,index,sensor_id,latitude,longitude
0,0,773869,34.15497,-118.31829
1,1,767541,34.11621,-118.23799
2,2,767542,34.11641,-118.23819
3,3,717447,34.07248,-118.26772
4,4,717446,34.07142,-118.26572


In [22]:
sensor_df=sensor_df.drop(columns=['index'],axis=1)

In [23]:
sensor_df.sample(5)

,sensor_id,latitude,longitude
9,717816,34.15562,-118.46860
48,760650,34.07505,-118.23256
11,767471,34.15691,-118.22469
42,773927,34.15262,-118.28034
58,773954,34.15544,-118.29344


In [24]:
sensor_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sensor_id  207 non-null    int64  
 1   latitude   207 non-null    float64
 2   longitude  207 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 5.0 KB


In [25]:
sensor_df.shape

(207, 3)

In [26]:
sensor_df['sensor_id'] = sensor_df['sensor_id'].astype('object')

In [27]:
sensor_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sensor_id  207 non-null    object 
 1   latitude   207 non-null    float64
 2   longitude  207 non-null    float64
dtypes: float64(2), object(1)
memory usage: 5.0+ KB


## Merging sensor location and traffic

In [28]:
traffic_geo = traffic_long.merge(
    sensor_df,
    on="sensor_id",
    how="left"
)

print("Traffic + sensor location shape:", traffic_geo.shape)
print("Missing latitude:", traffic_geo["latitude"].isna().sum())
print("Missing longitude:", traffic_geo["longitude"].isna().sum())

traffic_geo.head()

Traffic + sensor location shape: (6565709, 5)
Missing latitude: 6565709
Missing longitude: 6565709


,timestamp,sensor_id,speed,latitude,longitude
0,2012-03-01 00:00:00,773869,64.375000,NaN,NaN
1,2012-03-01 00:05:00,773869,62.666667,NaN,NaN
2,2012-03-01 00:10:00,773869,64.000000,NaN,NaN
3,2012-03-01 00:25:00,773869,57.333333,NaN,NaN
4,2012-03-01 00:30:00,773869,66.500000,NaN,NaN


In [29]:
print(sensor_df.head())
print(sensor_df.columns.tolist())
print(sensor_df.dtypes)

  sensor_id  latitude  longitude
0    773869  34.15497 -118.31829
1    767541  34.11621 -118.23799
2    767542  34.11641 -118.23819
3    717447  34.07248 -118.26772
4    717446  34.07142 -118.26572
['sensor_id', 'latitude', 'longitude']
sensor_id     object
latitude     float64
longitude    float64
dtype: object


In [30]:
print(traffic_long["sensor_id"].head())
print(traffic_long["sensor_id"].dtype)

0    773869
1    773869
2    773869
3    773869
4    773869
Name: sensor_id, dtype: object
object


In [31]:
sensor_clean = sensor_df.copy()

# Check if column names have extra spaces
sensor_clean.columns = sensor_clean.columns.str.strip()

print(sensor_clean.columns.tolist())
sensor_clean.head()

['sensor_id', 'latitude', 'longitude']


,sensor_id,latitude,longitude
0,773869,34.15497,-118.31829
1,767541,34.11621,-118.23799
2,767542,34.11641,-118.23819
3,717447,34.07248,-118.26772
4,717446,34.07142,-118.26572


In [32]:
traffic_sensor_ids = set(traffic_long["sensor_id"].astype(str).unique())

best_sensor_col = None
best_match_count = 0

for col in sensor_clean.columns:
    current_ids = set(sensor_clean[col].astype(str).str.strip().tolist())
    match_count = len(traffic_sensor_ids.intersection(current_ids))

    print(col, "matched:", match_count)

    if match_count > best_match_count:
        best_match_count = match_count
        best_sensor_col = col

print("Best sensor ID column:", best_sensor_col)
print("Best match count:", best_match_count)

sensor_id matched: 207
latitude matched: 0
longitude matched: 0
Best sensor ID column: sensor_id
Best match count: 207


In [33]:
# Standardize both sides again
traffic_long["sensor_id"] = traffic_long["sensor_id"].astype(str).str.strip()
sensor_df["sensor_id"] = sensor_df["sensor_id"].astype(str).str.strip()

# Make clean sensor dataframe
sensor_clean = sensor_df[["sensor_id", "latitude", "longitude"]].copy()

sensor_clean["latitude"] = pd.to_numeric(sensor_clean["latitude"], errors="coerce")
sensor_clean["longitude"] = pd.to_numeric(sensor_clean["longitude"], errors="coerce")

sensor_clean = sensor_clean.dropna(subset=["sensor_id", "latitude", "longitude"])
sensor_clean = sensor_clean.drop_duplicates(subset=["sensor_id"]).reset_index(drop=True)

print(sensor_clean.head())
print(sensor_clean.dtypes)

  sensor_id  latitude  longitude
0    773869  34.15497 -118.31829
1    767541  34.11621 -118.23799
2    767542  34.11641 -118.23819
3    717447  34.07248 -118.26772
4    717446  34.07142 -118.26572
sensor_id     object
latitude     float64
longitude    float64
dtype: object


In [34]:
traffic_sensor_ids = set(traffic_long["sensor_id"].unique())
location_sensor_ids = set(sensor_clean["sensor_id"].unique())

matched = traffic_sensor_ids.intersection(location_sensor_ids)

print("Traffic sensors:", len(traffic_sensor_ids))
print("Location sensors:", len(location_sensor_ids))
print("Matched sensors:", len(matched))
print("Unmatched traffic sensors:", len(traffic_sensor_ids - location_sensor_ids))

Traffic sensors: 207
Location sensors: 207
Matched sensors: 207
Unmatched traffic sensors: 0


In [35]:
traffic_geo = traffic_long.merge(
    sensor_clean,
    on="sensor_id",
    how="left"
)

print("Traffic + sensor location shape:", traffic_geo.shape)
print("Missing latitude:", traffic_geo["latitude"].isna().sum())
print("Missing longitude:", traffic_geo["longitude"].isna().sum())

traffic_geo.head()

Traffic + sensor location shape: (6565709, 5)
Missing latitude: 0
Missing longitude: 0


,timestamp,sensor_id,speed,latitude,longitude
0,2012-03-01 00:00:00,773869,64.375000,34.15497,-118.31829
1,2012-03-01 00:05:00,773869,62.666667,34.15497,-118.31829
2,2012-03-01 00:10:00,773869,64.000000,34.15497,-118.31829
3,2012-03-01 00:25:00,773869,57.333333,34.15497,-118.31829
4,2012-03-01 00:30:00,773869,66.500000,34.15497,-118.31829


In [36]:
traffic_geo["weather_time"] = traffic_geo["timestamp"].dt.floor("h")

In [37]:
traffic_geo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6565709 entries, 0 to 6565708
Data columns (total 6 columns):
 #   Column        Dtype         
---  ------        -----         
 0   timestamp     datetime64[ns]
 1   sensor_id     object        
 2   speed         float64       
 3   latitude      float64       
 4   longitude     float64       
 5   weather_time  datetime64[ns]
dtypes: datetime64[ns](2), float64(3), object(1)
memory usage: 300.6+ MB


## Weather Dataset

In [38]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/content/trafficpulse_dataset")
WEATHER_PATH = BASE_PATH / "weather"

weather_file = WEATHER_PATH / "la_weather_metr_la_period.csv"

weather_df = pd.read_csv(weather_file)

print("Weather dataset loaded successfully!")
print("Shape:", weather_df.shape)
print("\nColumns:")
print(weather_df.columns.tolist())

weather_df.head()

Weather dataset loaded successfully!
Shape: (2856, 9)

Columns:
['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'weather_code', 'cloud_cover', 'wind_speed_10m', 'wind_gusts_10m']


,time,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m
0,2012-03-01 00:00:00,6.8,94,0.0,0.0,0,11,3.6,12.6
1,2012-03-01 01:00:00,6.6,94,0.0,0.0,1,22,4.0,11.2
2,2012-03-01 02:00:00,6.3,95,0.0,0.0,2,70,4.0,11.5
3,2012-03-01 03:00:00,6.5,95,0.0,0.0,2,78,2.2,11.5
4,2012-03-01 04:00:00,6.4,96,0.0,0.0,2,78,1.6,11.2


In [39]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2856 entries, 0 to 2855
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  2856 non-null   object 
 1   temperature_2m        2856 non-null   float64
 2   relative_humidity_2m  2856 non-null   int64  
 3   precipitation         2856 non-null   float64
 4   rain                  2856 non-null   float64
 5   weather_code          2856 non-null   int64  
 6   cloud_cover           2856 non-null   int64  
 7   wind_speed_10m        2856 non-null   float64
 8   wind_gusts_10m        2856 non-null   float64
dtypes: float64(5), int64(3), object(1)
memory usage: 200.9+ KB


In [40]:
weather_df["time"] = pd.to_datetime(weather_df["time"])

weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2856 entries, 0 to 2855
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  2856 non-null   datetime64[ns]
 1   temperature_2m        2856 non-null   float64       
 2   relative_humidity_2m  2856 non-null   int64         
 3   precipitation         2856 non-null   float64       
 4   rain                  2856 non-null   float64       
 5   weather_code          2856 non-null   int64         
 6   cloud_cover           2856 non-null   int64         
 7   wind_speed_10m        2856 non-null   float64       
 8   wind_gusts_10m        2856 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(3)
memory usage: 200.9 KB


In [41]:
(weather_df == 0).sum()

,0
time,0
temperature_2m,0
relative_humidity_2m,0
precipitation,2723
rain,2725
weather_code,1929
cloud_cover,1322
wind_speed_10m,4
wind_gusts_10m,0


In [42]:
weather_df = (
    weather_df
    .drop_duplicates()
    .sort_values("time")
    .reset_index(drop=True)
)

weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2856 entries, 0 to 2855
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  2856 non-null   datetime64[ns]
 1   temperature_2m        2856 non-null   float64       
 2   relative_humidity_2m  2856 non-null   int64         
 3   precipitation         2856 non-null   float64       
 4   rain                  2856 non-null   float64       
 5   weather_code          2856 non-null   int64         
 6   cloud_cover           2856 non-null   int64         
 7   wind_speed_10m        2856 non-null   float64       
 8   wind_gusts_10m        2856 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(3)
memory usage: 200.9 KB


In [43]:
traffic_weather = traffic_geo.merge(
    weather_df,
    left_on="weather_time",
    right_on="time",
    how="left"
)

traffic_weather = traffic_weather.drop(columns=["time"])

print(traffic_weather.shape)
print(traffic_weather.isna().sum())
traffic_weather.head()

(6565709, 14)
timestamp               0
sensor_id               0
speed                   0
latitude                0
longitude               0
weather_time            0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
rain                    0
weather_code            0
cloud_cover             0
wind_speed_10m          0
wind_gusts_10m          0
dtype: int64


,timestamp,sensor_id,speed,latitude,longitude,weather_time,temperature_2m,relative_humidity_2m,precipitation,rain,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m
0,2012-03-01 00:00:00,773869,64.375000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,0,11,3.6,12.6
1,2012-03-01 00:05:00,773869,62.666667,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,0,11,3.6,12.6
2,2012-03-01 00:10:00,773869,64.000000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,0,11,3.6,12.6
3,2012-03-01 00:25:00,773869,57.333333,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,0,11,3.6,12.6
4,2012-03-01 00:30:00,773869,66.500000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,0,11,3.6,12.6


## Road Network Dataset

In [44]:
import geopandas as gpd
from pathlib import Path

BASE_PATH = Path("/content/content/trafficpulse_dataset")
OSM_PATH = BASE_PATH / "osm"

edges_path = OSM_PATH / "la_road_edges.geojson"
nodes_path = OSM_PATH / "la_road_nodes.geojson"

edges_gdf = gpd.read_file(edges_path)
nodes_gdf = gpd.read_file(nodes_path)

print("Edges shape:", edges_gdf.shape)
print("Nodes shape:", nodes_gdf.shape)

/usr/local/lib/python3.12/dist-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Edges shape: (117290, 18)
Nodes shape: (42613, 9)


In [45]:
print("Edge Columns:")
print(edges_gdf.columns.tolist())

print("\nNode Columns:")
print(nodes_gdf.columns.tolist())

edges_gdf.head()

Edge Columns:
['u', 'v', 'key', 'osmid', 'highway', 'lanes', 'maxspeed', 'name', 'oneway', 'reversed', 'length', 'bridge', 'ref', 'tunnel', 'access', 'width', 'junction', 'geometry']

Node Columns:
['osmid', 'y', 'x', 'highway', 'street_count', 'ref', 'junction', 'railway', 'geometry']


,u,v,key,osmid,highway,lanes,maxspeed,name,oneway,reversed,length,bridge,ref,tunnel,access,width,junction,geometry
0,653688,21300195,0,"[907145138, 895315641, 895315642, 398770659]",[secondary],[6],[35 mph],[National Boulevard],False,False,88.746024,None,None,None,None,None,None,"LINESTRING (-118.42955 34.02702, -118.42948 34..."
1,653688,1614655105,0,"[1376721881, 398770658, 759468526, 759468527]",[secondary],"[5, 6, 4]",[35 mph],[National Boulevard],False,True,102.260072,None,None,None,None,None,None,"LINESTRING (-118.42955 34.02702, -118.4298 34...."
2,653689,1711811908,0,398771138,[secondary],[5],[35 mph],[National Boulevard],False,False,19.377863,None,None,None,None,None,None,"LINESTRING (-118.42547 34.02864, -118.42538 34..."
3,653689,122734100,0,"[404964730, 398771139]",[secondary],"[5, 4]",[35 mph],[National Boulevard],False,True,109.728974,None,None,None,None,None,None,"LINESTRING (-118.42547 34.02864, -118.42556 34..."
4,653689,1711811906,0,643327598,[tertiary],None,None,[Military Avenue],False,False,19.493944,None,None,None,None,None,None,"LINESTRING (-118.42547 34.02864, -118.42554 34..."


In [46]:
print(edges_gdf.isnull().sum())

u                0
v                0
key              0
osmid            0
highway          0
lanes        71362
maxspeed     80225
name          2845
oneway           0
reversed         0
length           0
bridge      115484
ref         114971
tunnel      117182
access      116304
width       116796
junction    117000
geometry         0
dtype: int64


In [47]:
road_df = edges_gdf[
    [
        "osmid",
        "highway",
        "length",
        "lanes",
        "maxspeed",
        "oneway",
        "geometry"
    ]
].copy()

road_df.head()

,osmid,highway,length,lanes,maxspeed,oneway,geometry
0,"[907145138, 895315641, 895315642, 398770659]",[secondary],88.746024,[6],[35 mph],False,"LINESTRING (-118.42955 34.02702, -118.42948 34..."
1,"[1376721881, 398770658, 759468526, 759468527]",[secondary],102.260072,"[5, 6, 4]",[35 mph],False,"LINESTRING (-118.42955 34.02702, -118.4298 34...."
2,398771138,[secondary],19.377863,[5],[35 mph],False,"LINESTRING (-118.42547 34.02864, -118.42538 34..."
3,"[404964730, 398771139]",[secondary],109.728974,"[5, 4]",[35 mph],False,"LINESTRING (-118.42547 34.02864, -118.42556 34..."
4,643327598,[tertiary],19.493944,None,None,False,"LINESTRING (-118.42547 34.02864, -118.42554 34..."


In [48]:
road_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 117290 entries, 0 to 117289
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   osmid     117290 non-null  object  
 1   highway   117290 non-null  object  
 2   length    117290 non-null  float64 
 3   lanes     45928 non-null   object  
 4   maxspeed  37065 non-null   object  
 5   oneway    117290 non-null  bool    
 6   geometry  117290 non-null  geometry
dtypes: bool(1), float64(1), geometry(1), object(4)
memory usage: 5.5+ MB


In [49]:
road_df.isnull().sum()

,0
osmid,0
highway,0
length,0
lanes,71362
maxspeed,80225
oneway,0
geometry,0


In [50]:
road_df["highway"].value_counts().head(20)

,count
highway,
[residential],74718
[secondary],14067
[tertiary],13774
[primary],8900
[unclassified],2511
[motorway_link],1284
[motorway],1004
[primary_link],368
[secondary_link],257


In [51]:
road_df['geometry']

,geometry
0,"LINESTRING (-118.42955 34.02702, -118.42948 34..."
1,"LINESTRING (-118.42955 34.02702, -118.4298 34...."
2,"LINESTRING (-118.42547 34.02864, -118.42538 34..."
3,"LINESTRING (-118.42547 34.02864, -118.42556 34..."
4,"LINESTRING (-118.42547 34.02864, -118.42554 34..."
...,...
117285,"LINESTRING (-118.28678 34.07476, -118.28684 34..."
117286,"LINESTRING (-118.28678 34.07476, -118.28681 34..."
117287,"LINESTRING (-118.28678 34.07476, -118.28666 34..."
117288,"LINESTRING (-118.46502 34.0288, -118.46502 34...."


In [52]:
import geopandas as gpd

sensor_points = traffic_geo[["sensor_id", "latitude", "longitude"]].drop_duplicates()

sensor_gdf = gpd.GeoDataFrame(
    sensor_points,
    geometry=gpd.points_from_xy(sensor_points["longitude"], sensor_points["latitude"]),
    crs="EPSG:4326"
)

road_gdf = road_df.to_crs("EPSG:3857")
sensor_gdf = sensor_gdf.to_crs("EPSG:3857")

In [53]:
sensor_road = gpd.sjoin_nearest(
    sensor_gdf,
    road_gdf[["osmid", "highway", "length", "lanes", "maxspeed", "oneway", "geometry"]],
    how="left",
    distance_col="distance_to_road"
)

sensor_road = sensor_road.drop(columns=["geometry", "index_right"])

sensor_road.head()

,sensor_id,latitude,longitude,osmid,highway,length,lanes,maxspeed,oneway,distance_to_road
0,773869,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.664920,[4],[65 mph],True,2.615576
30777,767541,34.11621,-118.23799,863801551,[motorway],668.103761,[4],[65 mph],True,12.462052
62901,767542,34.11641,-118.23819,863796619,[motorway],1187.538510,[4],[65 mph],True,2.055587
95025,717447,34.07248,-118.26772,"[607788520, 31590106, 148252631]",[motorway],621.719397,[4],[55 mph],True,2.315838
127149,717446,34.07142,-118.26572,"[1419931202, 1419931203, 607788519, 1419931210...",[motorway],532.354893,[4],[55 mph],True,1.892949


In [54]:
sensor_road.shape

(207, 10)

In [55]:
final_df = traffic_weather.merge(
    sensor_road,
    on="sensor_id",
    how="left"
)

print(final_df.shape)
print(final_df.isna().sum())
final_df.head()

(6565709, 23)
timestamp                    0
sensor_id                    0
speed                        0
latitude_x                   0
longitude_x                  0
weather_time                 0
temperature_2m               0
relative_humidity_2m         0
precipitation                0
rain                         0
weather_code                 0
cloud_cover                  0
wind_speed_10m               0
wind_gusts_10m               0
latitude_y                   0
longitude_y                  0
osmid                        0
highway                      0
length                       0
lanes                    32124
maxspeed                351237
oneway                       0
distance_to_road             0
dtype: int64


,timestamp,sensor_id,speed,latitude_x,longitude_x,weather_time,temperature_2m,relative_humidity_2m,precipitation,rain,...,wind_gusts_10m,latitude_y,longitude_y,osmid,highway,length,lanes,maxspeed,oneway,distance_to_road
0,2012-03-01 00:00:00,773869,64.375000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.66492,[4],[65 mph],True,2.615576
1,2012-03-01 00:05:00,773869,62.666667,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.66492,[4],[65 mph],True,2.615576
2,2012-03-01 00:10:00,773869,64.000000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.66492,[4],[65 mph],True,2.615576
3,2012-03-01 00:25:00,773869,57.333333,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.66492,[4],[65 mph],True,2.615576
4,2012-03-01 00:30:00,773869,66.500000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",[motorway],626.66492,[4],[65 mph],True,2.615576


In [57]:
final_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Traffic/final_merged_dataset.csv")

KeyboardInterrupt: 

In [61]:
from pathlib import Path

OUTPUT_DIR = Path("/content/trafficpulse_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "traffic_weather_road.parquet"


In [62]:
final_df_clean = final_df.copy()

for col in final_df_clean.columns:
    if final_df_clean[col].dtype == "object":
        final_df_clean[col] = final_df_clean[col].astype(str)

final_df_clean.to_parquet(
    output_file,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Saved:", output_file)
print(final_df_clean.shape)

Saved: /content/trafficpulse_final/traffic_weather_road.parquet
(6565709, 23)


In [63]:
loaded_df = pd.read_parquet(output_file)

print(loaded_df.shape)
loaded_df.head()

(6565709, 23)


,timestamp,sensor_id,speed,latitude_x,longitude_x,weather_time,temperature_2m,relative_humidity_2m,precipitation,rain,...,wind_gusts_10m,latitude_y,longitude_y,osmid,highway,length,lanes,maxspeed,oneway,distance_to_road
0,2012-03-01 00:00:00,773869,64.375000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",['motorway'],626.66492,['4'],['65 mph'],True,2.615576
1,2012-03-01 00:05:00,773869,62.666667,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",['motorway'],626.66492,['4'],['65 mph'],True,2.615576
2,2012-03-01 00:10:00,773869,64.000000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",['motorway'],626.66492,['4'],['65 mph'],True,2.615576
3,2012-03-01 00:25:00,773869,57.333333,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",['motorway'],626.66492,['4'],['65 mph'],True,2.615576
4,2012-03-01 00:30:00,773869,66.500000,34.15497,-118.31829,2012-03-01,6.8,94,0.0,0.0,...,12.6,34.15497,-118.31829,"[156415224, 1453854234, 42420635]",['motorway'],626.66492,['4'],['65 mph'],True,2.615576
